## Measure inference performance of PyTorch model on CPU

First, we are going to measure the inference performance of an already-trained PyTorch model on CPU. After completing this section, you should understand:

-   how to measure the inference latency of a PyTorch model
-   how to measure the throughput of batch inference of a PyTorch model
-   how to compare eager model execution vs a compiled model

You will execute this notebook *in a Jupyter container running on a compute instance*, not on the general-purpose Chameleon Jupyter environment from which you provision resources.

#### install packages

In [50]:
!pip install pytorch-ignite

In [45]:
!pip install evaluate sacrebleu rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 147.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 153.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 189.2 MB/s eta 0:00:0000:01
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=440e84a14319c869be1c1438cd95d031c6432752a90bee581377109ef76d1922
  Stored in directory: /home/jovyan/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.0
    Uninstalling pyarrow-19.0.0:
      Successfully uninstalled pyarrow-19.0.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.9
    Uninstalling dill-

In [7]:
!pip install "optimum[onnxruntime]" transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 141.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 158.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 152.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 66.7 MB/s eta 0:00:00


In [10]:
!pip install accelerate

#### Load Models & Datasets

In [38]:
# runs in jupyter container on node-serve-model
import os
import torch
from torch.utils.data import DataLoader,Subset
from torchvision import datasets, transforms
import time
import numpy as np
import json
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import pandas as pd

Here we are importing the pretrained T5 model from Huggingface as a Pytorch model and the ONNX version

In [95]:
from pathlib import Path

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from optimum.onnxruntime import ORTModelForSeq2SeqLM
import onnxruntime as ort

model_id = "google-t5/t5-small"

session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL

tokenizer = AutoTokenizer.from_pretrained(model_id)

# onnx_model = ORTModelForSeq2SeqLM.from_pretrained(
#     model_id,
#     export=True,
#     providers=["CPUExecutionProvider"],
#     session_options=session_options,
# )

base_dir = Path("/home/jovyan/work/models")
pt_dir = base_dir / "pytorch"
onnx_dir = base_dir / "onnx"

pt_dir.mkdir(parents=True, exist_ok=True)
onnx_dir.mkdir(parents=True, exist_ok=True)

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# save PyTorch version
pt_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
# pt_model.save_pretrained(pt_dir)
# tokenizer.save_pretrained(pt_dir)

# # export and save ONNX version
# session_options = ort.SessionOptions()
# session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
# onnx_model = ORTModelForSeq2SeqLM.from_pretrained(
#     model_id, 
#     export=True,providers=["CPUExecutionProvider"],
#     session_options=session_options,)
# # onnx_model.save_pretrained(onnx_dir)
# # tokenizer.save_pretrained(onnx_dir)

# print("Saved PyTorch model to:", pt_dir)
# print("Saved ONNX model to:", onnx_dir)

In [96]:
# Setting the models to run on CPU  
device = torch.device("cpu")
pt_model = pt_model.to(device)
onnx_model = onnx_model.to(device)

# model = torch.load(model_path, map_location=device, weights_only=False)
# model.eval()  

In [97]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# Load your eval file
df = pd.read_json("/mnt/eval.jsonl", lines=True)

model_id = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)

max_input_len = 512
max_target_len = 256


class RecipeEvalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input_len=512, max_target_len=256):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Keep the "bad" input mostly intact
        input_text = row["input"].strip()
        target_text = row["target"].strip()

        # Optional: remove metadata noise only
        input_text = input_text.replace("<source:web_scrape>", "").strip()

        model_inputs = self.tokenizer(
            input_text,
            max_length=self.max_input_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        labels = self.tokenizer(
            text_target=target_text,
            max_length=self.max_target_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids = model_inputs["input_ids"].squeeze(0)
        attention_mask = model_inputs["attention_mask"].squeeze(0)
        labels = labels["input_ids"].squeeze(0)

        # Optional: ignore padding tokens in loss
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "raw_input": input_text,
            "raw_target": target_text,
        }


dataset = RecipeEvalDataset(df, tokenizer, max_input_len, max_target_len)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
)

# from torch.utils.data import Subset, DataLoader

subset_size = 200
small_dataset = Subset(dataset, range(subset_size))

small_dataloader = DataLoader(
    small_dataset,
    batch_size=4,
    shuffle=False,
)

batch = next(iter(small_dataloader))
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)
print(batch["raw_input"][0][:200])
print(batch["raw_target"][0][:200])

torch.Size([4, 512])
torch.Size([4, 512])
torch.Size([4, 256])
fix recipe: Title: Valentine's Thins
Ingredients: 36 RITZ Crackers with Whole Wheat | 2 pkg. (4 oz. each) BAKER'S Semi-Sweet Chocolate, broken into pieces, melted | 36 berry-shaped gummy candies (abou
Title: Valentine's Thins
Ingredients: 36 Ritz Crackers with Whole Wheat | 1 pkg. (225 g) Baker's Semi-Sweet Chocolate, melted | 36 Maynards Swedish Berries (about 2 pkg. [64 g each]) | 2 oz. Baker's W


and also prepare our test dataset:

### Pytorch Baseline Evaluations

We will measure:

-   the size of the model on disk
-   the latency when doing inference on single samples
-   the throughput when doing inference on batches of data
-   and the test accuracy

In [82]:
pt_model.compile()

#### Model size

We’ll start with model size. Our default `food11.pth` is a finetuned MobileNetV2, which is a small model designed for deployment on edge devices, so it is fairly small.

In [65]:
# runs in jupyter container on node-serve-model
model_path = os.path.join(pt_dir, "model.safetensors")
pytorch_model_size = os.path.getsize(model_path)
print(f"Pytorch Model Size on Disk: {pytorch_model_size/ (1e6) :.2f} MB")

Pytorch Model Size on Disk: 242.04 MB


#### Test accuracy

Next, we’ll measure the accuracy of this model on the test data

In [84]:
from tqdm.auto import tqdm

pt_model.eval()

predictions = []
references = []

for batch in tqdm(small_dataloader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        output_ids = pt_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
        )

    preds = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    predictions.extend(preds)
    references.extend(batch["raw_target"])

def normalize_text(s):
    return s.strip()

correct = sum(
    normalize_text(pred) == normalize_text(ref)
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(references)

print("Correct:", correct)
print("Total:", len(references))
print("Accuracy:", accuracy)

  0%|          | 0/50 [00:00<?, ?it/s]

Correct: 0
Total: 200
Accuracy: 0.0


In [85]:
from ignite.metrics import RougeL

m = RougeL(multiref="best")
# ["Rouge-L-F"]

m.reset()

for pred, ref in zip(predictions, references):
    candidate = pred.split()
    reference = ref.split()
    m.update(([candidate], [[reference]]))

rouge_l_f = m.compute()
print("Rouge Scores:", rouge_l_f)

Rouge Scores: {'Rouge-L-P': 0.46358754553454035, 'Rouge-L-R': 0.23677544389454494, 'Rouge-L-F': 0.23677544389454494}


#### Inference latency

Now, we’ll measure how long it takes the model to return a prediction for a single sample. We will run 100 trials, and then compute aggregate statistics.

In [98]:
import time
import torch

num_trials = 100
pt_model.eval()

batch = next(iter(small_dataloader))
input_ids = batch["input_ids"][0].unsqueeze(0)
attention_mask = batch["attention_mask"][0].unsqueeze(0)

with torch.no_grad():
    _ = pt_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
    )

latencies = []
with torch.no_grad():
    for _ in tqdm(range(num_trials)):
        start_time = time.perf_counter()
        _ = pt_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
        )
        latencies.append(time.perf_counter() - start_time)

print(f"Average latency: {sum(latencies)/len(latencies):.4f} sec")
print(f"Median latency: {sorted(latencies)[len(latencies)//2]:.4f} sec")
print(f"Min latency: {min(latencies):.4f} sec")
print(f"Max latency: {max(latencies):.4f} sec")

  0%|          | 0/100 [00:00<?, ?it/s]

Average latency: 1.2646 sec
Median latency: 1.2623 sec
Min latency: 1.2582 sec
Max latency: 1.3943 sec


In [99]:
# runs in jupyter container on node-serve-model
print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
print(f"Inference Throughput (single sample): {num_trials/np.sum(latencies):.2f} FPS")

Inference Latency (single sample, median): 1262.33 ms
Inference Latency (single sample, 95th percentile): 1269.22 ms
Inference Latency (single sample, 99th percentile): 1296.26 ms
Inference Throughput (single sample): 0.79 FPS


#### Batch throughput

Finally, we’ll measure the rate at which the model can return predictions for batches of data.

In [ ]:
# import time
# import torch
# from tqdm.auto import tqdm

num_batches = 50
pt_model.eval()

# Get one batch from your dataloader
batch = next(iter(small_dataloader))

input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]

batch_size = input_ids.shape[0]

# Warm-up run
with torch.no_grad():
    _ = pt_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
    )

batch_times = []
with torch.no_grad():
    for _ in tqdm(range(num_batches), desc="Batch throughput"):
        start_time = time.perf_counter()
        _ = pt_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
        )
        batch_times.append(time.perf_counter() - start_time)

avg_batch_time = sum(batch_times) / len(batch_times)
throughput = batch_size / avg_batch_time

print(f"Batch size: {batch_size}")
print(f"Average batch time: {avg_batch_time:.4f} sec")
print(f"Throughput: {throughput:.2f} samples/sec")
print(f"Min batch time: {min(batch_times):.4f} sec")
print(f"Max batch time: {max(batch_times):.4f} sec")

Batch throughput:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
# runs in jupyter container on node-serve-model
# batch_fps = (batch_input.shape[0] * num_batches) / np.sum(batch_times) 
# print(f"Batch Throughput: {batch_fps:.2f} FPS")

batch_fps = (input_ids.shape[0] * num_batches) / np.sum(batch_times)
print(f"Batch Throughput: {batch_fps:.2f} samples/sec")

#### Summary of Pytorch results

In [ ]:
# runs in jupyter container on node-serve-model
print(f"Model Size on Disk: {pytorch_model_size/ (1e6) :.2f} MB")
print(f"Rouge Scores:",rouge_l_f)
print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
print(f"Inference Throughput (single sample): {num_trials/np.sum(latencies):.2f} FPS")
print(f"Batch Throughput: {batch_fps:.2f} FPS")

<!-- 

compute_gigaio 

  Model name:             AMD EPYC 7763 64-Core Processor
    CPU family:           25
    Model:                1
    Thread(s) per core:   2
    Core(s) per socket:   64

-->
<!-- summary for mobilenet model

Model Size on Disk: 9.23 MB
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 60.16 ms
Inference Latency (single sample, 95th percentile): 77.22 ms
Inference Latency (single sample, 99th percentile): 77.37 ms
Inference Throughput (single sample): 15.82 FPS
Batch Throughput: 83.66 FPS


Model Size on Disk: 9.23 MB
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 73.97 ms
Inference Latency (single sample, 95th percentile): 83.16 ms
Inference Latency (single sample, 99th percentile): 83.94 ms
Inference Throughput (single sample): 13.34 FPS
Batch Throughput: 98.80 FPS

-->
<!-- summary for mobilenet compiled model

Model Size on Disk: 9.23 MB
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 26.92 ms
Inference Latency (single sample, 95th percentile): 49.79 ms
Inference Latency (single sample, 99th percentile): 64.55 ms
Inference Throughput (single sample): 32.35 FPS
Batch Throughput: 249.08 FPS

Model Size on Disk: 9.23 MB
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 34.14 ms
Inference Latency (single sample, 95th percentile): 53.85 ms
Inference Latency (single sample, 99th percentile): 60.23 ms
Inference Throughput (single sample): 27.39 FPS
Batch Throughput: 281.65 FPS

-->
<!-- 

(Intel CPU)

Model Size on Disk: 9.23 MB
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 12.69 ms
Inference Latency (single sample, 95th percentile): 12.83 ms
Inference Latency (single sample, 99th percentile): 12.97 ms
Inference Throughput (single sample): 78.73 FPS
Batch Throughput: 161.27 FPS

With compiling

Model Size on Disk: 9.23 MB
Accuracy: 90.59% (3032/3347 correct)
Inference Latency (single sample, median): 8.47 ms
Inference Latency (single sample, 95th percentile): 8.58 ms
Inference Latency (single sample, 99th percentile): 8.79 ms
Inference Throughput (single sample): 117.86 FPS
Batch Throughput: 474.67 FPS



-->

When you are done, download the fully executed notebook from the Jupyter container environment for later reference. (Note: because it is an executable file, and you are downloading it from a site that is not secured with HTTPS, you may have to explicitly confirm the download in some browsers.)

#### Eager mode execution vs compiled model

We had just evaluated a model in eager mode. However, in some (although, not all) cases we may get better performance from compiling the model into a graph, and executing it as a graph.

Go back to the cell where the model is loaded, and add

``` python
model.compile()
```

just below the call to `torch.load`. Then, run the notebook again (“Run \> Run All Cells”).

When you are done, download the fully executed notebook **again** from the Jupyter container environment for later reference.

### ONNX Evaluations

#### Model Size on disk

In [73]:
# runs in jupyter container on node-serve-model
model_path_1 = os.path.join(onnx_dir, "decoder_model.onnx")
model_path_2 = os.path.join(onnx_dir, "decoder_with_past_model.onnx")
model_path_3 = os.path.join(onnx_dir, "encoder_model.onnx")
model_size_1= os.path.getsize(model_path_1)
model_size_2= os.path.getsize(model_path_2)
model_size_3= os.path.getsize(model_path_3)
onnx_model_size = model_size_1+model_size_2+model_size_3
print(f"ONNX Model Size on Disk: {onnx_model_size/ (1e6) :.2f} MB")

ONNX Model Size on Disk: 593.79 MB


#### Model Accuracy

In [74]:
from tqdm.auto import tqdm

predictions = []
references = []

for batch in tqdm(small_dataloader, desc="ONNX evaluating"):
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    output_ids = onnx_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=256,
    )

    preds = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    predictions.extend(preds)
    references.extend(batch["raw_target"])

ONNX evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

In [75]:
from ignite.metrics import RougeL

m = RougeL(multiref="best")
# ["Rouge-L-F"]

m.reset()

for pred, ref in zip(predictions, references):
    candidate = pred.split()
    reference = ref.split()
    m.update(([candidate], [[reference]]))

rouge_l_f = m.compute()
print("Rouge Scores:", rouge_l_f)

Rouge Scores: {'Rouge-L-P': 0.46358754553454035, 'Rouge-L-R': 0.23677544389454494, 'Rouge-L-F': 0.23677544389454494}


#### Inference Latency

In [76]:
import time
from tqdm.auto import tqdm

num_trials = 100

batch = next(iter(small_dataloader))
input_ids = batch["input_ids"][0].unsqueeze(0)
attention_mask = batch["attention_mask"][0].unsqueeze(0)

# warm-up
_ = onnx_model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=128,
)

latencies = []
for _ in tqdm(range(num_trials), desc="ONNX single-sample latency"):
    start_time = time.perf_counter()
    _ = onnx_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
    )
    latencies.append(time.perf_counter() - start_time)

print(f"Average latency: {sum(latencies)/len(latencies):.4f} sec")
print(f"Median latency: {sorted(latencies)[len(latencies)//2]:.4f} sec")
print(f"Min latency: {min(latencies):.4f} sec")
print(f"Max latency: {max(latencies):.4f} sec")

ONNX single-sample latency:   0%|          | 0/100 [00:00<?, ?it/s]

Average latency: 1.2815 sec
Median latency: 1.2843 sec
Min latency: 1.2203 sec
Max latency: 1.4538 sec


In [77]:
# runs in jupyter container on node-serve-model
print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
print(f"Inference Throughput (single sample): {num_trials/np.sum(latencies):.2f} FPS")

Inference Latency (single sample, median): 1283.95 ms
Inference Latency (single sample, 95th percentile): 1332.92 ms
Inference Latency (single sample, 99th percentile): 1373.59 ms
Inference Throughput (single sample): 0.78 FPS


#### Batch Throughput

In [78]:
import time
from tqdm.auto import tqdm

num_batches = 50

batch = next(iter(small_dataloader))
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]

batch_size = input_ids.shape[0]

# warm-up
_ = onnx_model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=128,
)

batch_times = []
for _ in tqdm(range(num_batches), desc="ONNX batch throughput"):
    start_time = time.perf_counter()
    _ = onnx_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=128,
    )
    batch_times.append(time.perf_counter() - start_time)

avg_batch_time = sum(batch_times) / len(batch_times)
throughput = batch_size / avg_batch_time

print(f"Batch size: {batch_size}")
print(f"Average batch time: {avg_batch_time:.4f} sec")
print(f"Throughput: {throughput:.2f} samples/sec")
print(f"Min batch time: {min(batch_times):.4f} sec")
print(f"Max batch time: {max(batch_times):.4f} sec")

ONNX batch throughput:   0%|          | 0/50 [00:00<?, ?it/s]

Batch size: 4
Average batch time: 3.3722 sec
Throughput: 1.19 samples/sec
Min batch time: 3.1672 sec
Max batch time: 3.8771 sec


In [79]:
import numpy as np

batch_fps = (input_ids.shape[0] * num_batches) / np.sum(batch_times)
print(f"Batch Throughput: {batch_fps:.2f} samples/sec")

Batch Throughput: 1.19 samples/sec


#### Summary of ONNX Results

In [81]:

# runs in jupyter container on node-serve-model
print(f"Model Size on Disk: {onnx_model_size/ (1e6) :.2f} MB")
print(f"Rouge Scores:",rouge_l_f)
print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
print(f"Inference Throughput (single sample): {num_trials/np.sum(latencies):.2f} FPS")
print(f"Batch Throughput: {batch_fps:.2f} FPS")

Model Size on Disk: 593.79 MB
Rouge Scores: {'Rouge-L-P': 0.46358754553454035, 'Rouge-L-R': 0.23677544389454494, 'Rouge-L-F': 0.23677544389454494}
Inference Latency (single sample, median): 1283.95 ms
Inference Latency (single sample, 95th percentile): 1332.92 ms
Inference Latency (single sample, 99th percentile): 1373.59 ms
Inference Throughput (single sample): 0.78 FPS
Batch Throughput: 1.19 FPS
